# Week 1 – Machine Learning & AI Assignments

This notebook completes both uploaded Week 1 PDFs:

1. Load a CSV with Pandas and display the first 10 rows.
2. Explore data using `.info()` and `.describe()`.
3. Handle missing values using mean/median/mode imputation.
4. Encode categorical data using `LabelEncoder` and `OneHotEncoder`.
5. Split data into training and testing sets using Scikit-learn.
6. Train and evaluate a Linear Regression model.
7. Predict house prices from area.
8. Clean the Titanic dataset, encode `Sex` and `Embarked`, plot age distribution, and export a cleaned CSV.

**How to use:** Run the cells from top to bottom. This notebook works in Google Colab or Jupyter Notebook.

## 1. Install and import libraries

In [1]:
%pip -q install pandas numpy matplotlib seaborn scikit-learn

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

os.makedirs("datasets", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("visualizations", exist_ok=True)

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Create a small housing dataset

This sample dataset is included so the housing exercises run immediately without needing an extra download. It is illustrative practice data, not real-estate market data.

In [2]:
housing_data = {
    "area": [1000, 1200, 1500, 1800, 2000, 2200, 2500, 2800, 3000, 3200,
             1100, 1400, 1600, 1900, 2100, 2300, 2600, 2900, 3100, 3400],
    "bedrooms": [2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 2, np.nan, 3, 3, 4, 4, 4, 5, 5, 5],
    "location": ["Urban", "Suburban", "Urban", "Suburban", "Urban", "Rural",
                 "Urban", "Suburban", "Urban", "Rural", "Suburban", "Urban",
                 "Rural", "Urban", "Suburban", "Urban", "Rural", "Urban",
                 "Suburban", "Urban"],
    "price": [3000000, 3500000, 4500000, 5000000, 6000000, 5500000,
              7000000, 7500000, 8500000, 7800000, 3200000, 4200000,
              4000000, 5500000, 5800000, 6500000, 6200000, 8000000,
              8200000, 9000000]
}

housing_df = pd.DataFrame(housing_data)
housing_df.to_csv("datasets/housing.csv", index=False)

print("Housing dataset saved to datasets/housing.csv")
display(housing_df.head())

Housing dataset saved to datasets/housing.csv


,area,bedrooms,location,price
0,1000,2.0,Urban,3000000
1,1200,2.0,Suburban,3500000
2,1500,3.0,Urban,4500000
3,1800,3.0,Suburban,5000000
4,2000,4.0,Urban,6000000


## 3. Load CSV and explore the dataset (Practice Question 1)

In [3]:
df = pd.read_csv("datasets/housing.csv")

print("First 10 rows:")
display(df.head(10))

print("\\nDataset information:")
df.info()

print("\\nStatistical summary:")
display(df.describe())

print("\\nMissing values in each column:")
display(df.isnull().sum())

First 10 rows:


,area,bedrooms,location,price
0,1000,2.0,Urban,3000000
1,1200,2.0,Suburban,3500000
2,1500,3.0,Urban,4500000
3,1800,3.0,Suburban,5000000
4,2000,4.0,Urban,6000000
5,2200,4.0,Rural,5500000
6,2500,4.0,Urban,7000000
7,2800,5.0,Suburban,7500000
8,3000,5.0,Urban,8500000
9,3200,5.0,Rural,7800000


\nDataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   area      20 non-null     int64  
 1   bedrooms  19 non-null     float64
 2   location  20 non-null     object 
 3   price     20 non-null     int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 772.0+ bytes
\nStatistical summary:


,area,bedrooms,price
count,20.000000,19.000000,2.000000e+01
mean,2180.000000,3.789474,5.945000e+06
std,740.980858,1.084176,1.848605e+06
min,1000.000000,2.000000,3.000000e+06
25%,1575.000000,3.000000,4.425000e+06
50%,2150.000000,4.000000,5.900000e+06
75%,2825.000000,5.000000,7.575000e+06
max,3400.000000,5.000000,9.000000e+06


\nMissing values in each column:


,0
area,0
bedrooms,1
location,0
price,0


## 4. Handle missing values (mean and median imputation)

In [4]:
# Mean imputation for bedrooms (numerical column)
df["bedrooms"] = df["bedrooms"].fillna(df["bedrooms"].mean())

# Median imputation for area (shown as an example; this dataset has no missing area)
df["area"] = df["area"].fillna(df["area"].median())

print("Missing values after imputation:")
display(df.isnull().sum())

Missing values after imputation:


,0
area,0
bedrooms,0
location,0
price,0


## 5. Encode categorical variables (LabelEncoder and OneHotEncoder)

In [5]:
# Label Encoding: convert location categories into integer labels
label_encoder = LabelEncoder()
df["location_label"] = label_encoder.fit_transform(df["location"])

# One-Hot Encoding: create a separate binary column for each location category
try:
    one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
except TypeError:  # Compatibility with older scikit-learn versions
    one_hot_encoder = OneHotEncoder(sparse=False, handle_unknown="ignore")

encoded_array = one_hot_encoder.fit_transform(df[["location"]])
encoded_columns = one_hot_encoder.get_feature_names_out(["location"])
encoded_df = pd.DataFrame(encoded_array, columns=encoded_columns, index=df.index)

df_encoded = pd.concat([df.drop(columns=["location"]), encoded_df], axis=1)

print("Label-encoded and one-hot-encoded data:")
display(df_encoded.head())

df_encoded.to_csv("outputs/preprocessed_housing.csv", index=False)
print("Saved: outputs/preprocessed_housing.csv")

Label-encoded and one-hot-encoded data:


,area,bedrooms,price,location_label,location_Rural,location_Suburban,location_Urban
0,1000,2.0,3000000,2,0.0,0.0,1.0
1,1200,2.0,3500000,1,0.0,1.0,0.0
2,1500,3.0,4500000,2,0.0,0.0,1.0
3,1800,3.0,5000000,1,0.0,1.0,0.0
4,2000,4.0,6000000,2,0.0,0.0,1.0


Saved: outputs/preprocessed_housing.csv


## 6. Split dataset into training and testing sets (Practice Question 2)

In [6]:
# Use area to predict house price
X = df[["area"]]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 16
Testing samples: 4


## 7. Train Linear Regression and evaluate it (Practice Question 3)

In [7]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Linear Regression Evaluation")
print("----------------------------")
print(f"Mean Absolute Error (MAE): ₹{mae:,.2f}")
print(f"Mean Squared Error (MSE): ₹{mse:,.2f}")
print(f"Root Mean Squared Error (RMSE): ₹{rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

Linear Regression Evaluation
----------------------------
Mean Absolute Error (MAE): ₹196,848.74
Mean Squared Error (MSE): ₹54,333,157,969.07
Root Mean Squared Error (RMSE): ₹233,094.74
R² Score: 0.9874


## 8. Predict house price from area (Practice Question 4)

In [8]:
area_input = 2400  # Change this value to try another house area

new_house = pd.DataFrame({"area": [area_input]})
predicted_price = linear_model.predict(new_house)[0]

print(f"Estimated price for a {area_input:,} sq. ft. house: ₹{predicted_price:,.2f}")

Estimated price for a 2,400 sq. ft. house: ₹6,449,327.73


## 9. Titanic Survival Prediction – Data Cleaning Mini Project

The assignment specifies the Kaggle Titanic dataset. Download **train.csv** from the [Kaggle Titanic competition data page](https://www.kaggle.com/competitions/titanic/data), then upload it when prompted below.

In Google Colab, the upload window will appear. In local Jupyter, place `train.csv` in the same folder as this notebook or in `datasets/`.

In [ ]:
titanic_path = "datasets/titanic.csv"

if not os.path.exists(titanic_path):
    # Try the common local filename first
    if os.path.exists("train.csv"):
        titanic_path = "train.csv"
    else:
        try:
            from google.colab import files
            print("Please upload the Kaggle Titanic train.csv file.")
            uploaded = files.upload()
            uploaded_name = next(iter(uploaded))
            titanic_path = uploaded_name
        except ImportError:
            raise FileNotFoundError(
                "Titanic CSV not found. Download Kaggle train.csv and place it at "
                "'datasets/titanic.csv' or beside this notebook as 'train.csv'."
            )

titanic = pd.read_csv(titanic_path)

print("Original Titanic dataset:")
display(titanic.head())

print("\\nDataset information:")
titanic.info()

print("\\nMissing values before cleaning:")
display(titanic.isnull().sum())

Please upload the Kaggle Titanic train.csv file.


### 9.1 Handle missing values

In [ ]:
titanic_clean = titanic.copy()

# Median imputation for numerical columns
for column in ["Age", "Fare"]:
    if column in titanic_clean.columns:
        titanic_clean[column] = titanic_clean[column].fillna(
            titanic_clean[column].median()
        )

# Mode imputation for Embarked
if "Embarked" in titanic_clean.columns:
    titanic_clean["Embarked"] = titanic_clean["Embarked"].fillna(
        titanic_clean["Embarked"].mode()[0]
    )

# Cabin has a large number of missing values in the standard Kaggle dataset.
# Drop it to avoid retaining a mostly empty column.
if "Cabin" in titanic_clean.columns:
    titanic_clean = titanic_clean.drop(columns=["Cabin"])

print("Missing values after cleaning:")
display(titanic_clean.isnull().sum())

### 9.2 Encode Sex and Embarked

In [ ]:
# Label encode Sex
sex_encoder = LabelEncoder()
if "Sex" in titanic_clean.columns:
    titanic_clean["Sex"] = sex_encoder.fit_transform(titanic_clean["Sex"].astype(str))

# One-hot encode Embarked
if "Embarked" in titanic_clean.columns:
    try:
        embarked_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    except TypeError:
        embarked_encoder = OneHotEncoder(sparse=False, handle_unknown="ignore")

    embarked_array = embarked_encoder.fit_transform(titanic_clean[["Embarked"]])
    embarked_names = embarked_encoder.get_feature_names_out(["Embarked"])
    embarked_df = pd.DataFrame(
        embarked_array, columns=embarked_names, index=titanic_clean.index
    )
    titanic_clean = pd.concat(
        [titanic_clean.drop(columns=["Embarked"]), embarked_df], axis=1
    )

print("Encoded Titanic dataset:")
display(titanic_clean.head())

### 9.3 Visualise age distribution

In [ ]:
# Use the original age values to show the distribution before imputation.
plt.figure(figsize=(8, 5))
sns.histplot(data=titanic, x="Age", bins=30, kde=True)
plt.title("Age Distribution of Titanic Passengers")
plt.xlabel("Age")
plt.ylabel("Number of Passengers")
plt.tight_layout()
plt.savefig("visualizations/age_distribution.png", dpi=150)
plt.show()
print("Saved: visualizations/age_distribution.png")

### 9.4 Export cleaned Titanic dataset

In [ ]:
titanic_clean.to_csv("outputs/titanic_cleaned.csv", index=False)
print("Cleaned Titanic dataset saved to outputs/titanic_cleaned.csv")
print("Final dataset shape:", titanic_clean.shape)

## 10. Assignment completion summary

| Assignment requirement | Completed in |
|---|---|
| Load CSV and display first 10 rows | Sections 2–3 |
| `.info()` and `.describe()` | Section 3 |
| Mean/median imputation | Sections 4 and 9.1 |
| LabelEncoder and OneHotEncoder | Sections 5 and 9.2 |
| Train/test split | Section 6 |
| Linear Regression and evaluation | Section 7 |
| House price prediction from area | Section 8 |
| Titanic missing-data cleaning | Section 9.1 |
| Encode `Sex` and `Embarked` | Section 9.2 |
| Age distribution visualisation | Section 9.3 |
| Export cleaned CSV | Section 9.4 |

### Output files
- `outputs/preprocessed_housing.csv`
- `outputs/titanic_cleaned.csv`
- `visualizations/age_distribution.png`

**Note:** The notebook creates the housing sample data itself. The Titanic mini-project requires the Kaggle `train.csv` file.